# Neural BHE solver

This sections presents a the implementation of a neural solver of the Backward Heat Equation (BHE) written as a BSDE and solved with a neural solver under the Deep BSDE protocol.

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

Consider the multidimensional BHE, expressed as 
\begin{equation}
    \partial_t u + \frac{1}{2} \Delta u = 0,   u(T, x) = g(x) = |x|^2, x = \xi
\end{equation}
Here $\xi$ denotes the spatial value of interest and for which the system solves. $u(T, x)$ is the terminal value and is defined by the function $g(x)$. The PDE has exact solution $u = |x|^2 + d|T-t|$.

In this implementation we consider $d = 100$, $\xi = 0$, $T=1$, $t = 0$, $N = 20$ and $\Delta t = T/N$. Note that $u(0, 0) = 100$ and $\nabla u(t, x) = 2x.

In [ ]:
def heexact(t, x, T):
    # for t = 0 and x = 0 and T = 1 this will just be 'd'
    x = np.atleast_1d(x)
    d = x.shape[-1]
    return np.sum(x**2, axis=-1) + d*(T-t)

Similarly to MC the forward SDE over x can be written as $dX_t = \mu(t, X_t)dt + \sigma(t, X_t)dW_t = dW_t$ and $X_0 = \xi$
Under the Deep BSDE protocol, $u$ is represented as a stochastic process $Y_t = u(t, X_t)$ that for this equation can be derived as 
$dY_t = Z_t^TdW_t$ where $Y_T = g(X_T)$ for this particular equation. Here $Z_t$ is represented by a NN, this is the main differance from the MC method and is what makes the timestep necessary here, even though the terminal value of $x$ can be found explicitly. $Y_0$ here is also trainable and is the final value of the equation


The stochastic process can be integrated as $Y_{n +1} = Y_n - Z_n^T \Delta W_n$

In [ ]:
def W(dt, N, d):
    return np.sqrt(dt)*np.random.randn(N, d)

def g(x):
    return np.sum(x**2, axis= -1)

Each increment in time corresponds to a part of the neural network estimating $Z_t$. These are subnets to the full NN and are represented by a module. The full net ans consequently each subnet is updated after each batch of (in this case) 64 paths are run. The paths are run forward in time and starts at $Y_0 = \theta_y$ (a trainable parameter).

Intuitivly: the parameters of $Z_t$ (with trainables $\theta_Z$) and the the learnable parameter $Y_0 = \theta_y$ are initialized more or less randomly. This allows the equation to be "solved" (i.e forward integrated in time). As paths are collected (one batch) a measure of how good the parameters are can be formulated as the difference between the value of the attained $Y_T$ and the real terminal value of $g(X_T)$. This can formulate the loss function. The value of the loss function is used to update the model weights through backpropagation and, as the model trains, the loss will shrink, effectivly solving the PDE

The loss funtion can be stated as 
\begin{equation}
    \mathcal{L}(\theta) = \frac{1}{M} \sum_{m = 1}^M |Y_N^{(m)} - g(X_N^{(m)})|^2 = \mathbb{E}[|Y_T - g(X_T)|^2]
\end{equation}

In [ ]:
class SubNet(nn.Module):
    """ Subnet for each time increment Z_t, takes dimension and width as arguments.
    Typically, width = d + 10
    """
    def __init__(self, d, width):
        super().__init__()
        self.net = nn.Sequential(nn.BatchNorm1d(d),
            nn.Linear(d, width),
            nn.BatchNorm1d(d),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.BatchNorm1d(width),
            nn.ReLU(),
            nn.Linear(width, d),
        )
    def forward(self, x):
        return self.net(x)


class Znet():
    def __init__(self, d, dt, N, y0_init=(0., 1.)):
            super().__init__()
            self.N, self.dt, = N, dt
            self.y0 = nn.Parameter(torch.empty(1).uniform_(*y0_init))
            self.z0 = nn.Parameter(torch.empty(d).uniform_(-1,1))
            self.subnets = nn.ModuleList([SubNet(d, d+10) for _ in range(N-1)])
    def forward(self, X, dW):
        M = X.shape[0]
        y = self.y0.expand(M)
        z = self.z0.expand(M, -1)
        for n in range(self.N):
            t = n*self.dt
            y = y + self.dt*(z*dW[:,n]).sum(-1)
            if n < self.N-1:
                z = self.subnets[n](X[:, n + 1])
        return y


In [ ]:
T = 1.0
N = 20
dt = T/N


model = Znet(d=100, dt=dt, N = N)
opt = torch.optim.Adam(model.parameters(), lr = 1e-2)
